## 1. Generating Synthetic EHR Data with Synthea

As outlined in the project brief, **Synthea** is the recommended tool for generating synthetic patient EHR data. Synthea is a Java-based application that can simulate patient populations and generate realistic health records in various formats, including CSV.

To use Synthea, you typically need to:

1.  **Download Synthea:** Get the latest release from the [Synthea GitHub repository](https://github.com/synthetichealth/synthea).
2.  **Run Synthea:** Execute the Java application from your local machine or a dedicated server. You can configure parameters like the number of patients, population demographics, and output formats.
    ```bash
    # Example command to run Synthea (from its root directory)
    # Adjust parameters as needed, e.g., -p 1000 for 1000 patients
    ./run_synthea -p 1000
    ```
3.  **Collect Output:** Synthea will generate various CSV files (e.g., `patients.csv`, `conditions.csv`, `medications.csv`, `observations.csv`) in an `output/csv` directory.
4.  **Upload to Colab:** Once generated, you would upload these CSV files to your Colab environment (e.g., to a `data/raw` folder) for processing.

For the purpose of this notebook, we will assume you have generated a `patients.csv` file with the relevant static EHR data, and a `observations.csv` file which can provide some baseline measurements like HbA1c and fasting glucose. If you do not wish to run Synthea yourself, you can look for publicly available synthetic datasets that match the structure suggested in the brief.

## 2. Expected EHR Data Structure (Static Patient Data)

Based on the project brief's "STREAM A: Historical EHR" section (Section 13), a patient-level table (`patients.csv` or a combined view) should contain static information like:

-   `patient_id`
-   `age`
-   `sex`
-   `diabetes_status` (derived from conditions)
-   `hypertension_status` (derived from conditions)
-   `bmi` (can be calculated or derived from observations)
-   `hba1c` (from observations)
-   `fasting_glucose` (from observations)
-   `medication` (derived from medications data)
-   `previous_complications` (derived from conditions/observations)

In [18]:
import pandas as pd

# Placeholder for loading Synthea-generated EHR patient data
# In a real scenario, you would upload your Synthea output CSVs (e.g., patients.csv, observations.csv)
# and then preprocess them to create a consolidated static patient profile.

# For demonstration, let's create a dummy DataFrame that mimics the expected structure.
# You would replace this with actual loaded data from Synthea.

try:
    # Attempt to load actual data if available (e.g., after running Synthea and uploading)
    # This assumes 'patients.csv' and 'observations.csv' are in a 'data/raw' directory
    patients_df = pd.read_csv('data/raw/patients.csv')
    observations_df = pd.read_csv('data/raw/observations.csv')

    # Basic preprocessing to get some of the required static features
    # (This is a simplified example; actual Synthea output parsing would be more complex)
    ehr_df = patients_df[['Id', 'BIRTHDATE', 'GENDER']].copy()
    ehr_df.rename(columns={'Id': 'patient_id', 'BIRTHDATE': 'birthdate', 'GENDER': 'sex'}, inplace=True)
    ehr_df['age'] = (pd.to_datetime('now') - pd.to_datetime(ehr_df['birthdate'])).dt.days // 365

    # Example: derive diabetes status (highly simplified)
    # In a real Synthea output, you'd join with conditions.csv and look for 'Diabetes' codes
    ehr_df['diabetes_status'] = ehr_df['patient_id'].apply(lambda x: 'Yes' if int(x.split('-')[1]) % 2 == 0 else 'No')
    ehr_df['hypertension_status'] = ehr_df['patient_id'].apply(lambda x: 'Yes' if int(x.split('-')[1]) % 3 == 0 else 'No')

    # Example: add dummy HbA1c and BMI values
    ehr_df['hba1c'] = ehr_df['patient_id'].apply(lambda x: 5.5 + (int(x.split('-')[1]) % 5) * 0.5)
    ehr_df['bmi'] = ehr_df['patient_id'].apply(lambda x: 20 + (int(x.split('-')[1]) % 10) * 1.5)
    ehr_df['medication'] = ehr_df['patient_id'].apply(lambda x: 'Metformin' if ehr_df.loc[ehr_df['patient_id'] == x, 'diabetes_status'].iloc[0] == 'Yes' else 'None')

    # Select and display relevant columns
    static_ehr_df = ehr_df[['patient_id', 'age', 'sex', 'diabetes_status', 'hypertension_status', 'bmi', 'hba1c', 'medication']]
    print("Successfully loaded and processed EHR data (or created dummy data).")
    display(static_ehr_df.head())

except FileNotFoundError:
    print("Synthea data files not found. Creating a dummy EHR DataFrame for demonstration.")
    print("Please generate Synthea data and upload to 'data/raw/' or adapt the loading code.")
    # Create a dummy DataFrame if files are not found
    data = {
        'patient_id': [f'PATIENT-{i:04d}' for i in range(1, 6)],
        'age': [58, 62, 45, 70, 50],
        'sex': ['M', 'F', 'M', 'F', 'M'],
        'diabetes_status': ['Yes', 'No', 'Yes', 'No', 'Yes'],
        'hypertension_status': ['No', 'Yes', 'No', 'Yes', 'No'],
        'bmi': [29.7, 25.1, 31.0, 28.5, 26.9],
        'hba1c': [8.2, 5.9, 7.5, 6.1, 7.8],
        'medication': ['Metformin', 'None', 'Insulin', 'None', 'Metformin']
    }
    static_ehr_df = pd.DataFrame(data)
    display(static_ehr_df)


Synthea data files not found. Creating a dummy EHR DataFrame for demonstration.
Please generate Synthea data and upload to 'data/raw/' or adapt the loading code.


,patient_id,age,sex,diabetes_status,hypertension_status,bmi,hba1c,medication
0,PATIENT-0001,58,M,Yes,No,29.7,8.2,Metformin
1,PATIENT-0002,62,F,No,Yes,25.1,5.9,None
2,PATIENT-0003,45,M,Yes,No,31.0,7.5,Insulin
3,PATIENT-0004,70,F,No,Yes,28.5,6.1,None
4,PATIENT-0005,50,M,Yes,No,26.9,7.8,Metformin


## 3. Expected Dynamic Sensor Data Structure (Wearable/Time-Series)

Following the project brief's "STREAM B: Wearable/time-series" section (Section 13), the dynamic sensor data should be time-stamped and patient-specific, including metrics like:

-   `patient_id`
-   `timestamp`
-   `heart_rate`
-   `hrv` (heart rate variability)
-   `sleep_duration`
-   `steps`
-   `activity_level`
-   `glucose`

For the initial proof-of-concept, we will generate synthetic time-series data for a few patients over a short period to mimic continuous monitoring.

In [19]:
import numpy as np
import datetime

# Placeholder for loading dynamic sensor data
# In a real scenario, this would come from wearable devices or a simulation.

try:
    # Attempt to load actual dynamic data if available
    # This assumes a 'dynamic_sensor_data.csv' is in a 'data/raw' directory
    dynamic_sensor_df = pd.read_csv('data/raw/dynamic_sensor_data.csv')
    dynamic_sensor_df['timestamp'] = pd.to_datetime(dynamic_sensor_df['timestamp'])
    print("Successfully loaded dynamic sensor data.")
    display(dynamic_sensor_df.head())

except FileNotFoundError:
    print("Dynamic sensor data file not found. Generating dummy time-series data for demonstration.")
    print("Please upload your dynamic sensor data to 'data/raw/' or adapt the loading code.")

    # Generate dummy dynamic sensor data
    num_patients = len(static_ehr_df) # Use the patient IDs from the static EHR data
    patient_ids = static_ehr_df['patient_id'].tolist()

    start_date = datetime.datetime(2023, 10, 26, 0, 0, 0)
    end_date = datetime.datetime(2023, 10, 27, 23, 59, 0)
    time_interval_minutes = 15

    all_dynamic_data = []

    for p_id in patient_ids:
        current_date = start_date
        while current_date <= end_date:
            # Simulate some dynamic data points
            heart_rate = np.random.randint(60, 100)
            hrv = np.random.randint(20, 60)
            sleep_duration = round(np.random.uniform(4, 9), 1) if current_date.hour > 0 and current_date.hour < 6 else np.nan # Simulate sleep only at night
            steps = np.random.randint(0, 500) if current_date.hour > 6 and current_date.hour < 22 else 0
            activity_level = 'Low' if steps < 100 else ('Medium' if steps < 300 else 'High')

            # Simulate glucose with a general trend and some variability
            base_glucose = np.random.randint(90, 180)
            if 'Yes' in static_ehr_df[static_ehr_df['patient_id'] == p_id]['diabetes_status'].values:
                base_glucose += np.random.randint(30, 80) # Higher glucose for diabetic patients
            glucose = max(80, min(300, base_glucose + np.random.randint(-20, 20)))

            all_dynamic_data.append({
                'patient_id': p_id,
                'timestamp': current_date,
                'heart_rate': heart_rate,
                'hrv': hrv,
                'sleep_duration': sleep_duration,
                'steps': steps,
                'activity_level': activity_level,
                'glucose': glucose
            })
            current_date += datetime.timedelta(minutes=time_interval_minutes)

    dynamic_sensor_df = pd.DataFrame(all_dynamic_data)

    # Fill NaNs for sleep duration if needed for consistency or specific features
    dynamic_sensor_df['sleep_duration'] = dynamic_sensor_df['sleep_duration'].ffill().bfill() # Simple fill with updated syntax

    print(f"Generated dummy dynamic sensor data for {num_patients} patients over {(end_date - start_date).days + 1} day(s).")
    display(dynamic_sensor_df.head())


Dynamic sensor data file not found. Generating dummy time-series data for demonstration.
Please upload your dynamic sensor data to 'data/raw/' or adapt the loading code.
Generated dummy dynamic sensor data for 5 patients over 2 day(s).


,patient_id,timestamp,heart_rate,hrv,sleep_duration,steps,activity_level,glucose
0,PATIENT-0001,2023-10-26 00:00:00,84,56,4.8,0,Low,150
1,PATIENT-0001,2023-10-26 00:15:00,90,34,4.8,0,Low,178
2,PATIENT-0001,2023-10-26 00:30:00,70,50,4.8,0,Low,116
3,PATIENT-0001,2023-10-26 00:45:00,91,34,4.8,0,Low,207
4,PATIENT-0001,2023-10-26 01:00:00,93,35,4.8,0,Low,228


## 4. Data Fusion: Combining Static and Dynamic Data

Data fusion is the core concept of the Digital Twin, where we combine the patient's static historical profile with their continuously updating dynamic sensor data. This creates a rich, time-aware representation of each patient.

The goal is to merge `static_ehr_df` (which contains baseline patient characteristics) with `dynamic_sensor_df` (which has time-series measurements). We'll perform a merge operation based on `patient_id`.

In [20]:
import numpy as np

# Perform data fusion by merging static EHR data with dynamic sensor data
# This will add the static patient characteristics to each dynamic time-series record.

# Ensure both dataframes are sorted for potentially more efficient merging and subsequent operations
dynamic_sensor_df = dynamic_sensor_df.sort_values(by=['patient_id', 'timestamp']).reset_index(drop=True)
static_ehr_df = static_ehr_df.sort_values(by='patient_id').reset_index(drop=True)

# Merge the two DataFrames on 'patient_id'
# We use a left merge to keep all dynamic sensor data points and attach the corresponding static EHR data
patient_twin_df = pd.merge(
    dynamic_sensor_df,
    static_ehr_df,
    on='patient_id',
    how='left'
)

# Convert categorical features to numerical (e.g., for ML models later)
patient_twin_df['sex_encoded'] = patient_twin_df['sex'].map({'M': 0, 'F': 1})
patient_twin_df['diabetes_status_encoded'] = patient_twin_df['diabetes_status'].map({'No': 0, 'Yes': 1})
patient_twin_df['hypertension_status_encoded'] = patient_twin_df['hypertension_status'].map({'No': 0, 'Yes': 1})

# Convert activity_level to numerical if desired for ML
activity_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
patient_twin_df['activity_level_encoded'] = patient_twin_df['activity_level'].map(activity_mapping)

print("Data fusion complete. Displaying the first few rows of the combined patient_twin_df:")
display(patient_twin_df.head())
print(f"Shape of the combined DataFrame: {patient_twin_df.shape}")

# Display some information about the combined DataFrame to check for missing values or data types
print("\nInfo about combined DataFrame:")
patient_twin_df.info()

Data fusion complete. Displaying the first few rows of the combined patient_twin_df:


,patient_id,timestamp,heart_rate,hrv,sleep_duration,steps,activity_level,glucose,age,sex,diabetes_status,hypertension_status,bmi,hba1c,medication,sex_encoded,diabetes_status_encoded,hypertension_status_encoded,activity_level_encoded
0,PATIENT-0001,2023-10-26 00:00:00,84,56,4.8,0,Low,150,58,M,Yes,No,29.7,8.2,Metformin,0,1,0,0
1,PATIENT-0001,2023-10-26 00:15:00,90,34,4.8,0,Low,178,58,M,Yes,No,29.7,8.2,Metformin,0,1,0,0
2,PATIENT-0001,2023-10-26 00:30:00,70,50,4.8,0,Low,116,58,M,Yes,No,29.7,8.2,Metformin,0,1,0,0
3,PATIENT-0001,2023-10-26 00:45:00,91,34,4.8,0,Low,207,58,M,Yes,No,29.7,8.2,Metformin,0,1,0,0
4,PATIENT-0001,2023-10-26 01:00:00,93,35,4.8,0,Low,228,58,M,Yes,No,29.7,8.2,Metformin,0,1,0,0


Shape of the combined DataFrame: (960, 19)

Info about combined DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   patient_id                   960 non-null    object        
 1   timestamp                    960 non-null    datetime64[ns]
 2   heart_rate                   960 non-null    int64         
 3   hrv                          960 non-null    int64         
 4   sleep_duration               960 non-null    float64       
 5   steps                        960 non-null    int64         
 6   activity_level               960 non-null    object        
 7   glucose                      960 non-null    int64         
 8   age                          960 non-null    int64         
 9   sex                          960 non-null    object        
 10  diabetes_status              960 no

## 5. Feature Engineering

Feature engineering is a critical step in building effective machine learning models, especially with time-series data. It involves transforming raw data into features that better represent the underlying problem and improve model performance. For our Digital Twin predicting glucose spikes, we need features that capture not just the current state, but also recent trends, variability, and the relationship between different physiological parameters.

Key types of features we'll engineer for glucose spike prediction include:

*   **Lagged Features:** Values of a variable from previous time steps (e.g., glucose 15 minutes ago, 30 minutes ago).
*   **Rolling Window Statistics:** Aggregated statistics (mean, min, max, standard deviation) over a recent time window (e.g., average glucose over the last hour).
*   **Rate of Change (Slope):** How quickly a variable is changing (e.g., glucose trend over the last 30 minutes).
*   **Time-based Features:** Day of week, hour of day, and potentially more complex cyclical features.
*   **Interactions:** Combinations of static and dynamic features if deemed relevant.

In [21]:
import numpy as np

# Sort data by patient and timestamp to ensure correct lag and rolling calculations
patient_twin_df = patient_twin_df.sort_values(by=['patient_id', 'timestamp']).reset_index(drop=True)

# --- Glucose-specific Features ---

# 1. Lagged Glucose Values
# We'll create features for glucose levels at previous time steps (e.g., 15 min, 30 min, 1 hour ago)
# Assuming a 15-minute interval, a lag of 1 means 15 minutes ago, lag of 2 means 30 minutes ago, etc.
for lag in [1, 2, 4]: # 15min, 30min, 1 hour
    patient_twin_df[f'glucose_lag_{lag}'] = patient_twin_df.groupby('patient_id')['glucose'].shift(lag)

# 2. Rolling Mean and Standard Deviation of Glucose
# Captures recent glucose trends and variability
for window in [4, 8]: # 1 hour (4 * 15min), 2 hours (8 * 15min)
    patient_twin_df[f'glucose_rolling_mean_{window}'] = patient_twin_df.groupby('patient_id')['glucose'].transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    patient_twin_df[f'glucose_rolling_std_{window}'] = patient_twin_df.groupby('patient_id')['glucose'].transform(lambda x: x.rolling(window=window, min_periods=1).std())

# 3. Glucose Rate of Change (Slope)
# Simple difference over previous step
patient_twin_df['glucose_rate_of_change'] = patient_twin_df.groupby('patient_id')['glucose'].diff()

# --- Time-based Features (from timestamp) ---
patient_twin_df['hour_of_day'] = patient_twin_df['timestamp'].dt.hour
patient_twin_df['day_of_week'] = patient_twin_df['timestamp'].dt.dayofweek
patient_twin_df['is_weekend'] = patient_twin_df['day_of_week'].isin([5, 6]).astype(int)

# --- Other Dynamic Feature Interactions/Rolling Stats (Example for Heart Rate) ---
for window in [4]:
    patient_twin_df[f'hr_rolling_mean_{window}'] = patient_twin_df.groupby('patient_id')['heart_rate'].transform(lambda x: x.rolling(window=window, min_periods=1).mean())


print("Feature engineering complete. Displaying the first few rows with new features:")
display(patient_twin_df.head())
print(f"New shape of the DataFrame: {patient_twin_df.shape}")

# Check for NaNs introduced by lagging/rolling operations
print("\nNumber of NaNs after feature engineering (should only be at the beginning of each patient's series for lagged features):")
print(patient_twin_df.isnull().sum()[patient_twin_df.isnull().sum() > 0])

Feature engineering complete. Displaying the first few rows with new features:


,patient_id,timestamp,heart_rate,hrv,sleep_duration,steps,activity_level,glucose,age,sex,...,glucose_lag_4,glucose_rolling_mean_4,glucose_rolling_std_4,glucose_rolling_mean_8,glucose_rolling_std_8,glucose_rate_of_change,hour_of_day,day_of_week,is_weekend,hr_rolling_mean_4
0,PATIENT-0001,2023-10-26 00:00:00,84,56,4.8,0,Low,150,58,M,...,NaN,150.00,NaN,150.00,NaN,NaN,0,3,0,84.000000
1,PATIENT-0001,2023-10-26 00:15:00,90,34,4.8,0,Low,178,58,M,...,NaN,164.00,19.798990,164.00,19.798990,28.0,0,3,0,87.000000
2,PATIENT-0001,2023-10-26 00:30:00,70,50,4.8,0,Low,116,58,M,...,NaN,148.00,31.048349,148.00,31.048349,-62.0,0,3,0,81.333333
3,PATIENT-0001,2023-10-26 00:45:00,91,34,4.8,0,Low,207,58,M,...,NaN,162.75,38.896229,162.75,38.896229,91.0,0,3,0,83.750000
4,PATIENT-0001,2023-10-26 01:00:00,93,35,4.8,0,Low,228,58,M,...,150.0,182.25,48.692060,175.80,44.566804,21.0,1,3,0,86.000000


New shape of the DataFrame: (960, 31)

Number of NaNs after feature engineering (should only be at the beginning of each patient's series for lagged features):
glucose_lag_1              5
glucose_lag_2             10
glucose_lag_4             20
glucose_rolling_std_4      5
glucose_rolling_std_8      5
glucose_rate_of_change     5
dtype: int64


## 6. Creating the Prediction Label and Data Splitting

Before training a machine learning model, we need to:

1.  **Define the Target Variable (Label):** For early prediction of glucose spikes, we need to create a binary label (`0` or `1`) indicating whether a glucose spike occurs within a future time window. This requires careful consideration of what constitutes a 'spike' (e.g., a rapid increase, exceeding a threshold, or a combination).
2.  **Split the Data:** The fused and engineered data needs to be split into training, validation, and test sets. Given the time-series nature of our data, a random split is inappropriate as it could lead to data leakage (training on future data). Instead, we will perform a **time-aware split**, ensuring that our training data always precedes our validation and test data chronologically.

In [22]:
# --- 1. Define the Prediction Label (Glucose Spike) ---

# For this proof-of-concept, let's define a glucose spike as:
# 1. A future glucose value exceeding a certain threshold (e.g., 200 mg/dL).
# 2. AND/OR a significant increase from the current glucose level within a future time window (e.g., > 30 mg/dL in the next hour).

# Let's consider a spike to be an event where glucose rises above 180 mg/dL within the next 1 hour (4 time steps)
# We'll create a `future_glucose_max` column to look ahead.

prediction_window_steps = 4 # 1 hour ahead (4 * 15 minutes)
spike_threshold = 180 # mg/dL

# Calculate the maximum glucose value in the next `prediction_window_steps` for each patient
patient_twin_df['future_glucose_max'] = patient_twin_df.groupby('patient_id')['glucose'].transform(lambda x: x.rolling(window=prediction_window_steps, closed='right', min_periods=1).max().shift(-prediction_window_steps + 1))

# Define the label: 1 if future_glucose_max exceeds spike_threshold, else 0
patient_twin_df['glucose_spike_label'] = (patient_twin_df['future_glucose_max'] > spike_threshold).astype(int)

# Drop the `future_glucose_max` helper column
patient_twin_df = patient_twin_df.drop(columns=['future_glucose_max'])

# Drop rows where the label cannot be determined (at the end of each patient's series)
patient_twin_df.dropna(subset=['glucose_spike_label'], inplace=True)

print("Label creation complete. Displaying first few rows with the new 'glucose_spike_label':")
display(patient_twin_df.head())
print(f"Distribution of glucose spike labels: \n{patient_twin_df['glucose_spike_label'].value_counts()}")

# --- 2. Time-Aware Data Splitting ---

# We'll split the data based on timestamp, typically 70% train, 15% validation, 15% test.
# Find the cutoff timestamps for train/validation and validation/test splits.

# Get unique sorted timestamps
unique_timestamps = patient_twin_df['timestamp'].sort_values().unique()

train_split_idx = int(len(unique_timestamps) * 0.7)
val_split_idx = int(len(unique_timestamps) * 0.85)

train_cutoff_time = unique_timestamps[train_split_idx]
val_cutoff_time = unique_timestamps[val_split_idx]

# Perform the split
df_train = patient_twin_df[patient_twin_df['timestamp'] < train_cutoff_time]
df_val = patient_twin_df[(patient_twin_df['timestamp'] >= train_cutoff_time) & (patient_twin_df['timestamp'] < val_cutoff_time)]
df_test = patient_twin_df[patient_twin_df['timestamp'] >= val_cutoff_time]

print(f"\nData split complete: ")
print(f"Train set shape: {df_train.shape}")
print(f"Validation set shape: {df_val.shape}")
print(f"Test set shape: {df_test.shape}")

print("\nFirst few rows of the training set:")
display(df_train.head())

print("\nFirst few rows of the validation set:")
display(df_val.head())

print("\nFirst few rows of the test set:")
display(df_test.head())

# Check for any patient ID overlap to ensure no data leakage across splits if desired
# (though with time-based split, this is less of a concern for sequential models)
# patient_ids_train = set(df_train['patient_id'].unique())
# patient_ids_val = set(df_val['patient_id'].unique())
# patient_ids_test = set(df_test['patient_id'].unique())

# print(f"Overlap train-val: {len(patient_ids_train.intersection(patient_ids_val))}")
# print(f"Overlap train-test: {len(patient_ids_train.intersection(patient_ids_test))}")
# print(f"Overlap val-test: {len(patient_ids_val.intersection(patient_ids_test))}")

Label creation complete. Displaying first few rows with the new 'glucose_spike_label':


,patient_id,timestamp,heart_rate,hrv,sleep_duration,steps,activity_level,glucose,age,sex,...,glucose_rolling_mean_4,glucose_rolling_std_4,glucose_rolling_mean_8,glucose_rolling_std_8,glucose_rate_of_change,hour_of_day,day_of_week,is_weekend,hr_rolling_mean_4,glucose_spike_label
0,PATIENT-0001,2023-10-26 00:00:00,84,56,4.8,0,Low,150,58,M,...,150.00,NaN,150.00,NaN,NaN,0,3,0,84.000000,1
1,PATIENT-0001,2023-10-26 00:15:00,90,34,4.8,0,Low,178,58,M,...,164.00,19.798990,164.00,19.798990,28.0,0,3,0,87.000000,1
2,PATIENT-0001,2023-10-26 00:30:00,70,50,4.8,0,Low,116,58,M,...,148.00,31.048349,148.00,31.048349,-62.0,0,3,0,81.333333,1
3,PATIENT-0001,2023-10-26 00:45:00,91,34,4.8,0,Low,207,58,M,...,162.75,38.896229,162.75,38.896229,91.0,0,3,0,83.750000,1
4,PATIENT-0001,2023-10-26 01:00:00,93,35,4.8,0,Low,228,58,M,...,182.25,48.692060,175.80,44.566804,21.0,1,3,0,86.000000,1


Distribution of glucose spike labels: 
glucose_spike_label
1    614
0    346
Name: count, dtype: int64

Data split complete: 
Train set shape: (670, 32)
Validation set shape: (145, 32)
Test set shape: (145, 32)

First few rows of the training set:


,patient_id,timestamp,heart_rate,hrv,sleep_duration,steps,activity_level,glucose,age,sex,...,glucose_rolling_mean_4,glucose_rolling_std_4,glucose_rolling_mean_8,glucose_rolling_std_8,glucose_rate_of_change,hour_of_day,day_of_week,is_weekend,hr_rolling_mean_4,glucose_spike_label
0,PATIENT-0001,2023-10-26 00:00:00,84,56,4.8,0,Low,150,58,M,...,150.00,NaN,150.00,NaN,NaN,0,3,0,84.000000,1
1,PATIENT-0001,2023-10-26 00:15:00,90,34,4.8,0,Low,178,58,M,...,164.00,19.798990,164.00,19.798990,28.0,0,3,0,87.000000,1
2,PATIENT-0001,2023-10-26 00:30:00,70,50,4.8,0,Low,116,58,M,...,148.00,31.048349,148.00,31.048349,-62.0,0,3,0,81.333333,1
3,PATIENT-0001,2023-10-26 00:45:00,91,34,4.8,0,Low,207,58,M,...,162.75,38.896229,162.75,38.896229,91.0,0,3,0,83.750000,1
4,PATIENT-0001,2023-10-26 01:00:00,93,35,4.8,0,Low,228,58,M,...,182.25,48.692060,175.80,44.566804,21.0,1,3,0,86.000000,1



First few rows of the validation set:


,patient_id,timestamp,heart_rate,hrv,sleep_duration,steps,activity_level,glucose,age,sex,...,glucose_rolling_mean_4,glucose_rolling_std_4,glucose_rolling_mean_8,glucose_rolling_std_8,glucose_rate_of_change,hour_of_day,day_of_week,is_weekend,hr_rolling_mean_4,glucose_spike_label
134,PATIENT-0001,2023-10-27 09:30:00,91,24,5.2,250,Medium,234,58,M,...,174.00,40.323690,173.500,33.290925,73.0,9,4,0,80.50,1
135,PATIENT-0001,2023-10-27 09:45:00,84,38,5.2,344,High,204,58,M,...,187.75,38.282938,178.250,34.747045,-30.0,9,4,0,82.50,1
136,PATIENT-0001,2023-10-27 10:00:00,98,23,5.2,72,Low,188,58,M,...,196.75,30.521850,184.500,30.738529,-16.0,10,4,0,88.00,1
137,PATIENT-0001,2023-10-27 10:15:00,89,48,5.2,320,High,121,58,M,...,186.75,47.800802,177.750,38.156632,-67.0,10,4,0,90.50,1
138,PATIENT-0001,2023-10-27 10:30:00,90,44,5.2,411,High,166,58,M,...,169.75,36.040486,171.875,35.478112,45.0,10,4,0,90.25,1



First few rows of the test set:


,patient_id,timestamp,heart_rate,hrv,sleep_duration,steps,activity_level,glucose,age,sex,...,glucose_rolling_mean_4,glucose_rolling_std_4,glucose_rolling_mean_8,glucose_rolling_std_8,glucose_rate_of_change,hour_of_day,day_of_week,is_weekend,hr_rolling_mean_4,glucose_spike_label
163,PATIENT-0001,2023-10-27 16:45:00,77,29,5.2,6,Low,198,58,M,...,175.25,48.134361,171.625,33.161671,8.0,16,4,0,75.25,1
164,PATIENT-0001,2023-10-27 17:00:00,86,28,5.2,208,Medium,134,58,M,...,156.50,45.118363,168.625,35.568596,-64.0,17,4,0,78.50,1
165,PATIENT-0001,2023-10-27 17:15:00,87,22,5.2,499,High,176,58,M,...,174.50,28.489764,168.250,35.459232,42.0,17,4,0,83.25,1
166,PATIENT-0001,2023-10-27 17:30:00,97,43,5.2,51,Low,243,58,M,...,187.75,45.404662,175.875,44.295880,67.0,17,4,0,86.75,1
167,PATIENT-0001,2023-10-27 17:45:00,75,22,5.2,262,Medium,135,58,M,...,172.00,51.218486,173.625,46.046366,-108.0,17,4,0,86.25,1


## 7. Model Selection and Training

With our data prepared, labeled, and split, the next step is to select and train a machine learning model. For a proof-of-concept for early prediction of glucose spikes, we'll start with a classification model that can predict whether a spike (`1`) or no spike (`0`) will occur in the defined future window.

Key considerations for model selection in this context include:

*   **Interpretability:** Doctors need to understand *why* a prediction is made.
*   **Performance:** High accuracy, precision, and recall (especially for predicting rare events).
*   **Handling Time-Series Data:** Although we've engineered features, some models are inherently better at sequential data.
*   **Scalability:** Ability to handle larger patient populations and longer time series.

For our initial baseline, we'll use a relatively simple yet effective model, such as a **Logistic Regression** or a **Random Forest Classifier**, as they provide a good balance of performance and interpretability. We will train the model on `df_train`, evaluate it on `df_val`, and finally test its performance on the unseen `df_test` set.

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

# Define features (X) and target (y)
# We need to exclude patient_id, timestamp, and non-numeric/encoded features
# Also exclude any features that might be direct look-aheads into the future if they weren't dropped.

features_to_exclude = [
    'patient_id', 'timestamp', 'sex', 'diabetes_status', 'hypertension_status',
    'activity_level', 'medication', 'glucose_spike_label'
]

X_cols = [col for col in patient_twin_df.columns if col not in features_to_exclude]
y_col = 'glucose_spike_label'

# Separate features and target for each split
X_train = df_train[X_cols].copy()
y_train = df_train[y_col].copy()

X_val = df_val[X_cols].copy()
y_val = df_val[y_col].copy()

X_test = df_test[X_cols].copy()
y_test = df_test[y_col].copy()

# Handle NaNs introduced by feature engineering, typically filling with 0 or mean/median
# For simplicity, we'll fill with 0s for now, assuming NaNs usually appear at the start of a series.
# In a production system, more sophisticated imputation might be needed.
X_train.fillna(0, inplace=True)
X_val.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)

# Scale numerical features (important for many models, though less critical for Tree-based models like RF)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier inspection (optional, but good for consistent processing)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_cols, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_cols, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_cols, index=X_test.index)

print("Features and target prepared. NaNs filled and features scaled.")
print(f"X_train shape: {X_train_scaled.shape}, y_train shape: {y_train.shape}")

# --- Model Training: Random Forest Classifier ---

# Initialize a Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)

# Train the model
print("\nTraining Random Forest Classifier...")
rf_model.fit(X_train_scaled, y_train)
print("Training complete.")

# --- Model Evaluation on Validation Set ---
print("\nEvaluating model on validation set...")
y_val_pred = rf_model.predict(X_val_scaled)
y_val_proba = rf_model.predict_proba(X_val_scaled)[:, 1] # Probability of class 1

print("\nValidation Set Classification Report:")
print(classification_report(y_val, y_val_pred))
print(f"Validation AUC: {roc_auc_score(y_val, y_val_proba):.4f}")
print(f"Validation Precision: {precision_score(y_val, y_val_pred):.4f}")
print(f"Validation Recall: {recall_score(y_val, y_val_pred):.4f}")
print(f"Validation F1-score: {f1_score(y_val, y_val_pred):.4f}")

# Optional: Hyperparameter Tuning (e.g., using GridSearchCV)
# param_grid = {
#     'n_estimators': [50, 100, 200],
#     'max_depth': [None, 10, 20],
#     'min_samples_split': [2, 5]
# }
# grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
# grid_search.fit(X_train_scaled, y_train)
# print(f"Best parameters: {grid_search.best_params_}")
# best_rf_model = grid_search.best_estimator_

# For now, we'll stick with the initial model for demonstration.

Features and target prepared. NaNs filled and features scaled.
X_train shape: (670, 24), y_train shape: (670,)

Training Random Forest Classifier...
Training complete.

Evaluating model on validation set...

Validation Set Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.94      0.90        49
           1       0.97      0.93      0.95        96

    accuracy                           0.93       145
   macro avg       0.92      0.93      0.92       145
weighted avg       0.93      0.93      0.93       145

Validation AUC: 0.9409
Validation Precision: 0.9674
Validation Recall: 0.9271
Validation F1-score: 0.9468


## 8. Model Evaluation on Test Set

After training our model and tuning hyperparameters (if applicable) using the validation set, the final step in evaluating our model's predictive performance is to test it on the *unseen* test set. This provides an unbiased assessment of how the model is expected to perform on new, real-world data.

We will use the same evaluation metrics as for the validation set, including the classification report, AUC, precision, recall, and F1-score, to comprehensively understand the model's capabilities in identifying glucose spikes.

In [24]:
print("\nEvaluating model on test set...")
y_test_pred = rf_model.predict(X_test_scaled)
y_test_proba = rf_model.predict_proba(X_test_scaled)[:, 1] # Probability of class 1

print("\nTest Set Classification Report:")
print(classification_report(y_test, y_test_pred))
print(f"Test AUC: {roc_auc_score(y_test, y_test_proba):.4f}")
print(f"Test Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Test Recall: {recall_score(y_test, y_test_pred):.4f}")
print(f"Test F1-score: {f1_score(y_test, y_test_pred):.4f}")

print("\nModel evaluation on the test set complete. These metrics represent the expected performance of the Digital Twin on new, unseen patient data.")


Evaluating model on test set...

Test Set Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.82      0.85        57
           1       0.89      0.92      0.91        88

    accuracy                           0.88       145
   macro avg       0.88      0.87      0.88       145
weighted avg       0.88      0.88      0.88       145

Test AUC: 0.8788
Test Precision: 0.8901
Test Recall: 0.9205
Test F1-score: 0.9050

Model evaluation on the test set complete. These metrics represent the expected performance of the Digital Twin on new, unseen patient data.


## 9. Doctor-Facing Dashboard (Visualization and Explainability)

The ultimate goal of the Digital Twin is to provide actionable insights to healthcare professionals. A doctor-facing dashboard is essential for visualizing the model's predictions, understanding the contributing factors (explainability), and potentially simulating 'what-if' scenarios.

Key elements of the dashboard should include:

*   **Patient Overview:** Display of current vital signs and static EHR data.
*   **Prediction of Glucose Spikes:** Clear indication of predicted future spikes, perhaps with a probability score.
*   **Feature Importance/Explainability:** Tools to understand which features contributed most to a specific prediction (e.g., SHAP, LIME).
*   **Trend Visualization:** Time-series plots of glucose and other relevant physiological parameters.
*   **Interactivity:** Ability to select patients, time ranges, and potentially adjust parameters for scenario simulation.

For this proof-of-concept, we will focus on building a basic interactive visualization using `Plotly` and `Dash` (or similar) to display a patient's historical data, predicted glucose spikes, and potentially some key features contributing to those predictions.

In [25]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# For simplicity, let's consolidate the test data and predictions
# We'll use X_test (unscaled for display), y_test (true labels), and y_test_proba (predicted probabilities)

# Reconstruct the test dataframe with original features and predictions
df_test_display = df_test.copy()
df_test_display['predicted_spike_proba'] = y_test_proba
df_test_display['predicted_spike_label'] = y_test_pred

# Select a patient to visualize (e.g., the first patient in the test set)
patient_id_to_viz = df_test_display['patient_id'].iloc[0]
patient_data = df_test_display[df_test_display['patient_id'] == patient_id_to_viz]

print(f"Generating dashboard visualization for patient: {patient_id_to_viz}")

# Create subplots
fig = make_subplots(rows=3, cols=1,
                    shared_xaxes=True,
                    vertical_spacing=0.1,
                    subplot_titles=("Glucose Levels and Predicted Spikes",
                                    "Heart Rate & Activity Level",
                                    "Sleep Duration & HRV"))

# Plot 1: Glucose Levels and Predicted Spikes
fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['glucose'], mode='lines+markers', name='Glucose (actual)', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['predicted_spike_proba'], mode='lines', name='Spike Probability', line=dict(color='red', dash='dot')), row=1, col=1)

# Mark actual spikes (if y_test is available and aligned)
actual_spikes = patient_data[patient_data['glucose_spike_label'] == 1]
if not actual_spikes.empty:
    fig.add_trace(go.Scatter(x=actual_spikes['timestamp'], y=actual_spikes['glucose'], mode='markers', name='Actual Spike',
                             marker=dict(color='green', size=10, symbol='circle-open'),
                             hoverinfo='x+y+name'), row=1, col=1)

# Plot 2: Heart Rate & Activity Level
fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['heart_rate'], mode='lines', name='Heart Rate', line=dict(color='purple')), row=2, col=1)
fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['activity_level_encoded'], mode='lines', name='Activity Level (encoded)', line=dict(color='orange')), row=2, col=1)

# Plot 3: Sleep Duration & HRV
fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['sleep_duration'], mode='lines', name='Sleep Duration', line=dict(color='brown')), row=3, col=1)
fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['hrv'], mode='lines', name='HRV', line=dict(color='grey')), row=3, col=1)

# Update layout
fig.update_layout(height=800, title_text=f"Digital Twin Dashboard for Patient {patient_id_to_viz}", hovermode="x unified")
fig.update_xaxes(title_text="Timestamp", row=3, col=1)
fig.update_yaxes(title_text="Glucose (mg/dL) / Probability", row=1, col=1)
fig.update_yaxes(title_text="Heart Rate (bpm) / Activity Level", row=2, col=1)
fig.update_yaxes(title_text="Sleep (hours) / HRV", row=3, col=1)

fig.show()

print("\nThis visualization provides a basic example of how the Digital Twin's predictions and relevant physiological data can be presented in a doctor-facing dashboard. Further enhancements could include more interactive elements, detailed explainability, and multi-patient views.")

Generating dashboard visualization for patient: PATIENT-0001



This visualization provides a basic example of how the Digital Twin's predictions and relevant physiological data can be presented in a doctor-facing dashboard. Further enhancements could include more interactive elements, detailed explainability, and multi-patient views.


## 10. GitHub Repository Setup and Project Structure

Now that we have a working proof-of-concept for our Digital Twin, setting up a well-structured GitHub repository is crucial for managing the project, enabling collaboration, and ensuring reproducibility.

Here’s a recommended structure for your `TwinCare-Glyco` repository:

```
TwinCare-Glyco/
├── data/
│   ├── raw/                 # Raw, immutable data (e.g., Synthea output CSVs)
│   └── processed/           # Processed, engineered feature sets (e.g., patient_twin_df)
├── notebooks/
│   ├── 01_data_generation_and_fusion.ipynb  # This notebook
│   ├── 02_feature_engineering_and_labeling.ipynb
│   └── 03_model_training_and_dashboard.ipynb
├── src/
│   ├── data_processing.py   # Functions for data loading, cleaning, and fusion
│   ├── feature_engineering.py # Functions for creating features
│   ├── models.py            # Model definition, training, and prediction logic
│   └── dashboard_app.py     # Code for a more elaborate dashboard application (e.g., Dash/Streamlit)
├── models/
│   └── trained_model.pkl    # Saved trained machine learning models
├── reports/
│   └── figures/
│   └── presentations/
├── README.md                # Project overview, setup, and usage instructions
├── requirements.txt         # List of Python dependencies
├── .gitignore               # Files/directories to ignore (e.g., data/raw, models/, .ipynb_checkpoints/)
└── LICENSE
```

### Steps to Set Up Your GitHub Repository:

1.  **Create a New Repository:** Go to [GitHub](https://github.com/) and create a new public or private repository (e.g., `TwinCare-Glyco`).
2.  **Clone the Repository:** Clone the empty repository to your local machine (or in Colab using `!git clone <repo_url>` and then `!mv TwinCare-Glyco/* .` to move contents to current dir if desired).
3.  **Add Project Structure:** Create the directories as outlined above (`data`, `notebooks`, `src`, `models`, `reports`).
4.  **Save Notebooks and Code:**
    *   Download your current Colab notebook (`File > Download > Download .ipynb`).
    *   Rename it appropriately (e.g., `01_data_generation_and_fusion.ipynb`).
    *   Consider refactoring code cells into Python scripts in the `src/` directory for better organization and reusability (e.g., data loading functions into `src/data_processing.py`). This is generally good practice for larger projects.
    *   Upload the notebook and any refactored `.py` files to their respective directories in your cloned repository.
5.  **Create `requirements.txt`:** List all Python libraries used in your project (e.g., `pandas`, `numpy`, `scikit-learn`, `plotly`, `dash`). You can generate a basic one using `!pip freeze > requirements.txt` (though you'll want to clean it up).
6.  **Write `README.md`:** Provide a clear overview of your project, its goals, how to set it up, how to run the code, and how to interpret the results.
7.  **Add `.gitignore`:** Include common files and directories to ignore (e.g., `*.pyc`, `__pycache__/`, `.ipynb_checkpoints/`, `data/raw/`, `models/`).
8.  **Commit and Push:** Add all your files, commit your changes, and push them to your GitHub repository.

This structured approach will make your Digital Twin project professional, maintainable, and easy for others (and your future self!) to understand and contribute to.

### Creating an Interactive Dash Dashboard

To build a truly interactive doctor-facing dashboard, we'll use the **Dash** framework. Dash applications are web servers that run on Flask and communicate JSON packets to front-end React components. It allows for highly customizable interactive dashboards purely in Python.

This will be a basic example demonstrating:
1.  A dropdown to select a patient.
2.  Plots displaying the selected patient's glucose levels, predicted spike probabilities, and other vital signs.

**Note:** Running a full Dash application within a Colab notebook directly for live interaction can be challenging and often requires port forwarding or using `jupyter_dash`. For simplicity, the following code will set up the Dash application, and you would typically run it in a separate Python script locally or deploy it to a server.

In [31]:
# First, install Dash and its dependencies if you haven't already
# Removed jupyter_dash as it's deprecated and causing issues.
!pip install dash dash_core_components dash_html_components ngrok pyngrok

import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# --- ngrok setup for Colab --- (Crucial for external access)
from google.colab import userdata
# Get your ngrok auth token from https://dashboard.ngrok.com/auth/your-authtoken
# and save it as a Colab secret named 'NGROK_AUTH_TOKEN'
# If you don't have one, you might need to sign up for ngrok.
try:
    ngrok_token = userdata.get('NGROK_AUTH_TOKEN')
    if ngrok_token:
        # Authenticate ngrok
        !ngrok authtoken $ngrok_token
    else:
        print("NGROK_AUTH_TOKEN Colab secret not found. Please add your ngrok auth token to Colab secrets for external access.")
        print("You can still run the app, but it might not be externally accessible.")
except Exception as e:
    print(f"Error accessing NGROK_AUTH_TOKEN: {e}. You may need to set it up in Colab secrets.")

# Reconstruct the test dataframe with original features and predictions for the dashboard
# This step ensures we have all necessary columns for the dashboard to display.
# If df_test_display already exists, we use it; otherwise, we create it.

if 'df_test_display' not in locals():
    df_test_display = df_test.copy()
    df_test_display['predicted_spike_proba'] = y_test_proba
    df_test_display['predicted_spike_label'] = y_test_pred

# Get a list of unique patient IDs for the dropdown
patient_ids_for_dropdown = df_test_display['patient_id'].unique()

# Initialize the Dash app (using standard Dash for broader compatibility)
app = dash.Dash(__name__)

# Define the app layout
app.layout = html.Div([
    html.H1("TwinCare-Glyco: Digital Twin Dashboard"),

    html.Div([
        html.Label("Select Patient:"),
        dcc.Dropdown(
            id='patient-dropdown',
            options=[{'label': pid, 'value': pid} for pid in patient_ids_for_dropdown],
            value=patient_ids_for_dropdown[0] # Default selected patient
        ),
    ]),

    dcc.Graph(id='patient-data-graph')
])

# Define callback to update graph based on dropdown selection
@app.callback(
    Output('patient-data-graph', 'figure'),
    [Input('patient-dropdown', 'value')]
)
def update_graph(selected_patient_id):
    patient_data = df_test_display[df_test_display['patient_id'] == selected_patient_id]

    fig = make_subplots(rows=3, cols=1,
                        shared_xaxes=True,
                        vertical_spacing=0.1,
                        subplot_titles=("Glucose Levels and Predicted Spikes",
                                        "Heart Rate & Activity Level",
                                        "Sleep Duration & HRV"))

    # Plot 1: Glucose Levels and Predicted Spikes
    fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['glucose'], mode='lines+markers', name='Glucose (actual)', line=dict(color='blue')), row=1, col=1)
    fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['predicted_spike_proba'], mode='lines', name='Spike Probability', line=dict(color='red', dash='dot')), row=1, col=1)

    # Mark actual spikes
    actual_spikes = patient_data[patient_data['glucose_spike_label'] == 1]
    if not actual_spikes.empty:
        fig.add_trace(go.Scatter(x=actual_spikes['timestamp'], y=actual_spikes['glucose'], mode='markers', name='Actual Spike',
                                 marker=dict(color='green', size=10, symbol='circle-open'),
                                 hoverinfo='x+y+name'), row=1, col=1)

    # Plot 2: Heart Rate & Activity Level
    fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['heart_rate'], mode='lines', name='Heart Rate', line=dict(color='purple')), row=2, col=1)
    # Check if 'activity_level_encoded' column exists before plotting (defensive programming)
    if 'activity_level_encoded' in patient_data.columns:
        fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['activity_level_encoded'], mode='lines', name='Activity Level (encoded)', line=dict(color='orange')), row=2, col=1)
    else:
        fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=[None]*len(patient_data), mode='lines', name='Activity Level (encoded) [N/A]', line=dict(color='orange', dash='dash')), row=2, col=1)

    # Plot 3: Sleep Duration & HRV
    fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['sleep_duration'], mode='lines', name='Sleep Duration', line=dict(color='brown')), row=3, col=1)
    fig.add_trace(go.Scatter(x=patient_data['timestamp'], y=patient_data['hrv'], mode='lines', name='HRV', line=dict(color='grey')), row=3, col=1)

    fig.update_layout(height=800, title_text=f"Digital Twin Dashboard for Patient {selected_patient_id}", hovermode="x unified")
    fig.update_xaxes(title_text="Timestamp", row=3, col=1)
    fig.update_yaxes(title_text="Glucose (mg/dL) / Probability", row=1, col=1)
    fig.update_yaxes(title_text="Heart Rate (bpm) / Activity Level", row=2, col=1)
    fig.update_yaxes(title_text="Sleep (hours) / HRV", row=3, col=1)

    return fig

print("Dash app setup complete. To run, save this code to a .py file (e.g., `src/dashboard_app.py`)")
print("and execute `python src/dashboard_app.py` in your local environment.")
print("To run the app directly in Colab and access it externally, you need an ngrok authtoken.")
print("The app will now attempt to run, and if ngrok is configured, a public URL will be provided.")

# Run the app in a separate thread so Colab cell doesn't block indefinitely
from threading import Thread
from pyngrok import ngrok
import time

def run_dash_app():
    # use_reloader=False is important when running in a thread to prevent multiple instances
    app.run(port=8050, debug=True, use_reloader=False) # Changed from app.run_server to app.run

dash_thread = Thread(target=run_dash_app)
dash_thread.daemon = True # Allow the thread to exit when the main program exits
dash_thread.start()

# Give the server some time to start up
time.sleep(5)

# Get the ngrok public URL
try:
    # Ensure ngrok tunnel is established
    public_url = ngrok.connect(addr=8050, proto='http')
    print(f"\nDash app running at: {public_url}")
    print("Click the link above to access your interactive dashboard!")
except Exception as e:
    print(f"\nCould not establish ngrok tunnel: {e}")
    print("Please ensure ngrok is installed and authenticated, or run the app locally.")


Error accessing NGROK_AUTH_TOKEN: Secret NGROK_AUTH_TOKEN does not exist.. You may need to set it up in Colab secrets.
Dash app setup complete. To run, save this code to a .py file (e.g., `src/dashboard_app.py`)
and execute `python src/dashboard_app.py` in your local environment.
To run the app directly in Colab and access it externally, you need an ngrok authtoken.
The app will now attempt to run, and if ngrok is configured, a public URL will be provided.


<IPython.core.display.Javascript object>

ERROR:pyngrok.process.ngrok:t=2026-09-20T11:35:08+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"



Could not establish ngrok tunnel: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.
Please ensure ngrok is installed and authenticated, or run the app locally.


## 11. Saving Model Artifacts for Deployment and GitHub Migration

To ensure reproducibility and enable deployment, it's crucial to save the trained machine learning model and any preprocessing objects (like the `StandardScaler`) that were fitted on the training data. These artifacts will be saved to the `models/` directory as specified in the proposed GitHub structure.

**For the notebook itself, please download it manually via `File > Download > Download .ipynb` to save the current state.**

In [32]:
import os
import joblib

# Define the directory to save models
models_dir = 'models/'
os.makedirs(models_dir, exist_ok=True)

# Save the trained Random Forest model
model_path = os.path.join(models_dir, 'rf_model.joblib')
joblib.dump(rf_model, model_path)
print(f"Trained Random Forest model saved to: {model_path}")

# Save the StandardScaler used for feature scaling
scaler_path = os.path.join(models_dir, 'scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f"StandardScaler saved to: {scaler_path}")

print("\nModel artifacts are now saved and ready for migration to your GitHub repository and potential deployment.")

Trained Random Forest model saved to: models/rf_model.joblib
StandardScaler saved to: models/scaler.joblib

Model artifacts are now saved and ready for migration to your GitHub repository and potential deployment.
